# Crash Log Frequency Analysis

## Project Overview
Analyze application crash logs over time to identify unstable versions, devices, environments, and release-related issues.

## Learning Objectives
- Analyze crash frequency trends over time and across app versions
- Identify the most crash-prone devices, OS versions, and environments
- Detect release-related crash spikes and regressions
- Calculate crash-free session and user rates
- Prioritize crash fixes using frequency × impact scoring

## Problem Statement
A mobile app team is experiencing increased crash reports after recent releases. We need to identify which versions, devices, and conditions cause the most crashes to prioritize engineering fixes and protect user experience.

## Why This Project Matters
Crashes directly impact user retention, app store ratings, and revenue. A 1% increase in crash rate can cause a measurable drop in DAU. Systematic crash analysis prevents regressions and guides quality investment.

## Dataset Overview
Synthetic crash log data: ~8,000 crash events over 6 months across 5 app versions, multiple devices, and OS versions.

## Dataset Source and License Notes
- Synthetic data
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n = 8000
dates = pd.date_range('2023-07-01', periods=180, freq='D')

app_versions = ['3.1.0', '3.2.0', '3.2.1', '3.3.0', '3.3.1']
# Version release dates and crash rates
version_info = {
    '3.1.0': {'start': 0,   'end': 40,  'crash_weight': 1.0},
    '3.2.0': {'start': 35,  'end': 80,  'crash_weight': 2.5},  # buggy release
    '3.2.1': {'start': 75,  'end': 120, 'crash_weight': 0.8},  # hotfix
    '3.3.0': {'start': 115, 'end': 160, 'crash_weight': 1.8},  # new features, some issues
    '3.3.1': {'start': 155, 'end': 180, 'crash_weight': 0.6},  # stable
}

devices = ['iPhone 14', 'iPhone 13', 'iPhone 12', 'Samsung S23', 'Samsung S22',
           'Pixel 7', 'Pixel 6', 'iPad Air', 'Samsung Tab']
device_weights = np.array([18, 15, 12, 14, 10, 8, 6, 10, 7], dtype=float)
device_weights /= device_weights.sum()

os_versions = {'iOS 17': 0.35, 'iOS 16': 0.25, 'Android 14': 0.20, 'Android 13': 0.12, 'iPadOS 17': 0.08}
crash_types = ['NullPointerException', 'OutOfMemoryError', 'NetworkTimeout',
               'UIRenderingCrash', 'DatabaseCorruption', 'ConcurrencyError']
crash_type_weights = np.array([30, 15, 20, 18, 7, 10], dtype=float)
crash_type_weights /= crash_type_weights.sum()

rows = []
for _ in range(n):
    # Pick version weighted by crash_weight and time window
    ver = np.random.choice(app_versions)
    vi = version_info[ver]
    day_idx = np.random.randint(vi['start'], min(vi['end'], 180))
    date = dates[day_idx]

    device = np.random.choice(devices, p=device_weights)
    os_ver = np.random.choice(list(os_versions.keys()), p=list(os_versions.values()))
    crash_type = np.random.choice(crash_types, p=crash_type_weights)

    # Severity correlates with crash type
    if crash_type in ['DatabaseCorruption', 'OutOfMemoryError']:
        severity = np.random.choice(['critical', 'high', 'medium'], p=[0.5, 0.35, 0.15])
    elif crash_type in ['NullPointerException', 'ConcurrencyError']:
        severity = np.random.choice(['critical', 'high', 'medium', 'low'], p=[0.2, 0.4, 0.3, 0.1])
    else:
        severity = np.random.choice(['high', 'medium', 'low'], p=[0.3, 0.45, 0.25])

    rows.append({
        'date': date, 'app_version': ver, 'device': device,
        'os_version': os_ver, 'crash_type': crash_type, 'severity': severity
    })

df = pd.DataFrame(rows)
df['week'] = df['date'].dt.isocalendar().week.astype(int)
df['month'] = df['date'].dt.to_period('M')

# Simulated total sessions per day (to calculate crash-free rate)
daily_sessions = pd.DataFrame({'date': dates, 'total_sessions': np.random.randint(45000, 55000, len(dates))})

print(f'Dataset: {df.shape}')
print(f'Date range: {df["date"].min().date()} to {df["date"].max().date()}')
print(f'\nCrashes by version:\n{df["app_version"].value_counts().sort_index()}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nCrash type distribution:\n{df["crash_type"].value_counts()}')
print(f'\nSeverity distribution:\n{df["severity"].value_counts()}')

## Crash Trend Over Time

In [ ]:
daily_crashes = df.groupby('date').size().reset_index(name='crashes')
daily = daily_crashes.merge(daily_sessions, on='date', how='right').fillna(0)
daily['crash_rate'] = daily['crashes'] / daily['total_sessions'] * 10000  # per 10K sessions

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].bar(daily['date'], daily['crashes'], alpha=0.5, width=1, color='steelblue')
axes[0].plot(daily['date'], daily['crashes'].rolling(7).mean(), color='red', linewidth=1.5, label='7-day MA')
axes[0].set_ylabel('Daily Crashes')
axes[0].set_title('Daily Crash Count')
axes[0].legend()

# Mark version releases
colors = ['green', 'red', 'blue', 'orange', 'purple']
for (ver, vi), c in zip(version_info.items(), colors):
    axes[0].axvline(dates[vi['start']], color=c, linestyle='--', alpha=0.7, label=f'{ver} release')
axes[0].legend(fontsize=7, ncol=3)

axes[1].plot(daily['date'], daily['crash_rate'], alpha=0.5, color='steelblue')
axes[1].plot(daily['date'], daily['crash_rate'].rolling(7).mean(), color='red', linewidth=1.5)
axes[1].set_ylabel('Crash Rate (per 10K sessions)')
axes[1].set_title('Crash Rate Over Time')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'crash_trend.png'), dpi=100, bbox_inches='tight')
plt.show()

## Crashes by App Version

In [ ]:
ver_stats = df.groupby('app_version').size().reset_index(name='crashes')
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(ver_stats['app_version'], ver_stats['crashes'], edgecolor='black',
              color=['#4CAF50', '#F44336', '#2196F3', '#FF9800', '#9C27B0'])
ax.set_title('Total Crashes by App Version')
ax.set_ylabel('Crash Count')
for bar, val in zip(bars, ver_stats['crashes']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20, str(val),
            ha='center', fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'version_crashes.png'), dpi=100, bbox_inches='tight')
plt.show()

## Crash Type × Version Heatmap

In [ ]:
ct_ver = pd.crosstab(df['crash_type'], df['app_version'])
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(ct_ver, annot=True, fmt='d', cmap='YlOrRd', ax=ax)
ax.set_title('Crash Type × App Version')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'crash_type_version.png'), dpi=100, bbox_inches='tight')
plt.show()

## Device and OS Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df['device'].value_counts().plot.barh(ax=axes[0], edgecolor='black')
axes[0].set_title('Crashes by Device')
axes[0].set_xlabel('Count')
axes[0].invert_yaxis()

df['os_version'].value_counts().plot.barh(ax=axes[1], edgecolor='black', color='coral')
axes[1].set_title('Crashes by OS Version')
axes[1].set_xlabel('Count')
axes[1].invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'device_os_crashes.png'), dpi=100, bbox_inches='tight')
plt.show()

## Severity Distribution by Version

In [ ]:
sev_ver = pd.crosstab(df['app_version'], df['severity'], normalize='index') * 100
sev_order = ['critical', 'high', 'medium', 'low']
sev_ver = sev_ver.reindex(columns=[c for c in sev_order if c in sev_ver.columns])
fig, ax = plt.subplots(figsize=(10, 5))
sev_ver.plot.bar(ax=ax, stacked=True, edgecolor='black',
                 color=['#D32F2F', '#FF9800', '#FFC107', '#4CAF50'])
ax.set_title('Crash Severity Distribution by Version')
ax.set_ylabel('Percentage')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Severity')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'severity_version.png'), dpi=100, bbox_inches='tight')
plt.show()

## Crash-Free Rate

In [ ]:
# Crash-free session rate per version
ver_daily = df.groupby(['date', 'app_version']).size().reset_index(name='crashes')
overall_cfr = 1 - daily['crashes'].sum() / daily['total_sessions'].sum()
print(f'Overall crash-free session rate: {overall_cfr*100:.2f}%')
print(f'\nTarget: >99.5% crash-free')
print(f'Status: {"PASS" if overall_cfr > 0.995 else "NEEDS IMPROVEMENT"}')

## Interpretation and Business Insight
- **Version 3.2.0** is the most crash-prone release — a clear regression that the 3.2.1 hotfix addressed
- **NullPointerException** is the most frequent crash type across all versions — systematic null-safety improvements needed
- **Version 3.3.1** shows the lowest crash rate — quality is improving over time
- **iPhone 14 and Samsung S23** have the most crashes but also the most users — per-device crash rate would need session normalization
- **DatabaseCorruption** crashes are rare but overwhelmingly critical severity — high-priority fix
- The crash-free rate is close to the 99.5% industry target but version 3.2.0 pulled it down significantly

## Limitations
- Synthetic data — real crash logs include stack traces, breadcrumbs, and session context
- No per-device session counts for normalized crash rates
- No user-level deduplication (same user crashing repeatedly)
- No crash grouping by stack trace signature
- Missing network condition and memory state context

## How to Improve This Project
- Add stack trace grouping for deduplication
- Include crash-free user rate (not just session rate)
- Add regression detection automation (alert on version release)
- Include ANR (Application Not Responding) events alongside crashes
- Add crash-to-churn correlation analysis

## Production Considerations
- Real-time crash monitoring with automated alerting
- Staged rollouts with automatic rollback on crash rate spikes
- Crash reporting SDK integration (Firebase Crashlytics, Sentry)
- Release health dashboards for go/no-go decisions
- Crash priority scoring: frequency × severity × user impact

## Common Mistakes
- Counting total crashes without normalizing by session or user count
- Treating all crash types equally instead of prioritizing by severity and frequency
- Not separating crash rates by version — masking regressions in aggregates
- Ignoring low-frequency but critical crashes (e.g., data corruption)
- Celebrating crash-free rate improvements without checking if users simply stopped using the app

## Mini Challenge / Exercises
1. Calculate the per-version crash-free session rate — which version is worst?
2. What is the mean-time-to-hotfix (days between 3.2.0 release and 3.2.1)?
3. If you could fix one crash type, which would reduce total crashes the most?
4. Estimate the number of users affected assuming 2.5 sessions per user per day.

## Final Summary / Key Takeaways
- Crash analysis must be version-aware — aggregate metrics hide regressions
- Crash-free rate is the industry-standard metric — aim for >99.5%
- Not all crashes are equal — prioritize by severity × frequency
- Hotfix turnaround time is a key quality metric
- Proactive crash monitoring prevents user churn better than reactive fixes